<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/llm/ollama_gemma4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ollama — Gemma 4

[Gemma 4](https://blog.google/technology/google-deepmind/google-gemma-4/) is Google DeepMind's latest open model family. Key upgrades over earlier Gemma versions:

- **Multimodal** — accepts text **and image** inputs natively
- **128K context window** — handles long documents out of the box
- **Function calling / tool use** — OpenAI-compatible tool spec
- Available via [Ollama](https://ollama.com/library/gemma4) as `gemma4:12b` (and other sizes)

This notebook covers:
1. Text completions & chat
2. Streaming
3. **Multimodal (vision)** — image + text
4. **RAG** — local document Q&A with `VectorStoreIndex`
5. **Tool / function calling**
6. Structured outputs

## Setup

1. Install [Ollama](https://ollama.com) and start the server: `ollama serve`
2. Pull the model: `ollama pull gemma4:12b`
3. Install the Python packages below.

In [ ]:
!pip install llama-index-llms-ollama llama-index

In [ ]:
from llama_index.llms.ollama import Ollama

llm = Ollama(model="gemma4:12b", request_timeout=120.0)

## 1. Text completions & chat

In [ ]:
resp = llm.complete("Explain transformer attention in two sentences.")
print(resp)

In [ ]:
from llama_index.core.llms import ChatMessage

messages = [
    ChatMessage(role="system", content="You are a concise technical assistant."),
    ChatMessage(role="user", content="What makes Gemma 4 different from Gemma 2?"),
]
resp = llm.chat(messages)
print(resp)

## 2. Streaming

In [ ]:
response = llm.stream_complete("Write a short poem about local AI models.")
for r in response:
    print(r.delta, end="", flush=True)

In [ ]:
messages = [
    ChatMessage(role="system", content="You are a helpful assistant."),
    ChatMessage(role="user", content="List three benefits of running LLMs locally."),
]
resp = llm.stream_chat(messages)
for r in resp:
    print(r.delta, end="", flush=True)

## 3. Multimodal (vision) — image + text

Gemma 4 can reason about images. Pass an `ImageBlock` alongside a `TextBlock` in the user message.

In [ ]:
from llama_index.core.base.llms.types import ImageBlock, TextBlock

# Load an image from disk
with open("image.png", "rb") as f:
    image_data = f.read()

response = llm.chat(
    [
        ChatMessage(
            role="user",
            blocks=[
                ImageBlock(image=image_data),
                TextBlock(text="Describe what you see in this image in detail."),
            ],
        )
    ]
)
print(response)

In [ ]:
# You can also reference an image by URL
response = llm.chat(
    [
        ChatMessage(
            role="user",
            blocks=[
                ImageBlock(url="https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png"),
                TextBlock(text="What objects are in this image?"),
            ],
        )
    ]
)
print(response)

### Practical multimodal example: extract data from a receipt or document

In [ ]:
# Replace "receipt.png" with any invoice, receipt, or chart image
with open("receipt.png", "rb") as f:
    receipt_data = f.read()

response = llm.chat(
    [
        ChatMessage(
            role="user",
            blocks=[
                ImageBlock(image=receipt_data),
                TextBlock(text="Extract all line items and the total as JSON."),
            ],
        )
    ]
)
print(response)

## 4. RAG — local document Q&A

Gemma 4's 128K context window makes it well-suited for RAG over local files. Nothing leaves your machine.

In [ ]:
!pip install llama-index-embeddings-huggingface

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Use a small local embedding model — no API key required
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.llm = llm

# Load documents from a local folder (PDFs, .txt, .md, etc.)
documents = SimpleDirectoryReader("./data").load_data()
index = VectorStoreIndex.from_documents(documents)

query_engine = index.as_query_engine()
response = query_engine.query("Summarize the main points of these documents.")
print(response)

In [ ]:
# Multi-turn chat over your documents
chat_engine = index.as_chat_engine()
response = chat_engine.chat("What are the key dates mentioned?")
print(response)

response = chat_engine.chat("Can you give me more detail on the first one?")
print(response)

### RAG with streaming

In [ ]:
streaming_engine = index.as_query_engine(streaming=True)
streaming_response = streaming_engine.query("What are the action items from these documents?")
streaming_response.print_response_stream()

## 5. Tool / function calling

Gemma 4 supports the OpenAI-compatible tool spec used by Ollama.

In [ ]:
from typing import Annotated
from llama_index.core.tools import FunctionTool


def get_token_price(
    symbol: Annotated[str, "The token symbol, e.g. BTC, ETH"],
) -> str:
    """Return the current USD price of a crypto token (mock)."""
    prices = {"BTC": "$67,420", "ETH": "$3,510", "SOL": "$148"}
    return prices.get(symbol.upper(), f"Price for {symbol} not available.")


def summarize_wallet(
    address: Annotated[str, "Ethereum wallet address"],
) -> str:
    """Return a brief summary of a wallet's holdings (mock)."""
    return f"Wallet {address[:8]}… holds 1.2 ETH, 500 USDC, and 3 NFTs."


tools = [
    FunctionTool.from_defaults(fn=get_token_price),
    FunctionTool.from_defaults(fn=summarize_wallet),
]

response = llm.chat_with_tools(
    tools,
    user_msg="What is the price of ETH right now?",
)
tool_calls = llm.get_tool_calls_from_response(response)

for tc in tool_calls:
    print(f"Tool called: {tc.tool_name}({tc.tool_kwargs})")
    # Execute the tool
    for tool in tools:
        if tool.metadata.name == tc.tool_name:
            result = tool(**tc.tool_kwargs)
            print(f"Result: {result.raw_output}")

### ReAct agent with tools

In [ ]:
from llama_index.core.agent.workflow import ReActAgent

agent = ReActAgent(tools=tools, llm=llm)

response = await agent.run(
    "What is the ETH price? Also summarize wallet 0xAbCdEf1234567890."
)
print(response)

## 6. Structured outputs

Attach a Pydantic model to get guaranteed structured JSON from Gemma 4.

In [ ]:
from llama_index.core.bridge.pydantic import BaseModel, Field
from llama_index.core.llms import ChatMessage


class CryptoAnalysis(BaseModel):
    """A brief analysis of a cryptocurrency."""

    symbol: str = Field(description="Token symbol, e.g. BTC")
    sentiment: str = Field(description="bullish, bearish, or neutral")
    key_risk: str = Field(description="The single biggest risk factor")
    one_line_summary: str = Field(description="One sentence market summary")


sllm = llm.as_structured_llm(CryptoAnalysis)

response = sllm.chat(
    [ChatMessage(role="user", content="Give me an analysis of Ethereum.")]
)
analysis = response.raw
print(f"Symbol: {analysis.symbol}")
print(f"Sentiment: {analysis.sentiment}")
print(f"Key risk: {analysis.key_risk}")
print(f"Summary: {analysis.one_line_summary}")

## 7. Async usage

All methods have async variants for use in async applications.

In [ ]:
resp = await llm.acomplete("What is the capital of France?")
print(resp)

resp = await llm.achat(
    [ChatMessage(role="user", content="Name three open-source LLMs.")]
)
print(resp)

In [ ]:
# Async streaming
async for chunk in await llm.astream_complete("List five uses for multimodal AI:"):
    print(chunk.delta, end="", flush=True)

## Model sizes

| Ollama model | Parameters | Approx. RAM | Notes |
|---|---|---|---|
| `gemma4:2b` | 2B | ~2 GB | Fastest, lightest |
| `gemma4:12b` | 12B | ~8 GB | Best quality/speed balance |
| `gemma4:27b` | 27B | ~18 GB | Highest quality |

All sizes support multimodal input and function calling.

## Further reading

- [Ollama Gemma 4 page](https://ollama.com/library/gemma4)
- [Google DeepMind Gemma 4 announcement](https://blog.google/technology/google-deepmind/google-gemma-4/)
- [LlamaIndex Ollama integration docs](https://docs.llamaindex.ai/en/stable/examples/llm/ollama/)
- [LlamaIndex RAG guide](https://docs.llamaindex.ai/en/stable/understanding/rag/)